Execute the next two cells to open the visualization in the right window and to setup the demo.
Once the demo is setup three text fields with queries that exemplify the three steps of the law of task achieving body motion appear.
You can execute a query with the "Run Query" button and ask for the next solution with the "Next Solution" button until it prints "No more solutions".
Some queries produce visualized outputs in the right window, also the query text can be changed if you like.

In [1]:
%%bash --bg
rviz -d /home/jovyan/giskard_examples/launch/rvizweb_config/bmp.rviz

In [2]:
from demo import setup_demo
setup_demo()

In [12]:
from pyswip import Prolog

# Prolog() shares the same global SWI-Prolog engine that setup_demo() filled with kb.pl
prolog = Prolog()

def clean(x):
    if isinstance(x, bytes):
        return x.decode()
    if isinstance(x, list):
        return [clean(i) for i in x]
    return x

def show(query, limit=20):
    print("Query:", query)
    sols = list(prolog.query(query, maxresult=limit))
    for i, sol in enumerate(sols, 1):
        print(i, {k: clean(v) for k, v in sol.items()})
    if not sols:
        print("No solutions.")
    return sols

In [13]:
sols = show("taskRequest(Request, StateChange).", limit=50)
print("\nNumber of request/state-change pairs:", len(sols))

Query: taskRequest(Request, StateChange).
1 {'Request': 'open_fridge', 'StateChange': 'open(fridge)'}
2 {'Request': 'close_anything', 'StateChange': 'close(drawer_03)'}
3 {'Request': 'close_anything', 'StateChange': 'close(drawer_05)'}
4 {'Request': 'close_anything', 'StateChange': 'close(oven_drawer)'}
5 {'Request': 'close_anything', 'StateChange': 'close(drawer_04)'}
6 {'Request': 'close_anything', 'StateChange': 'close(drawer_01)'}
7 {'Request': 'close_anything', 'StateChange': 'close(drawer_02)'}
8 {'Request': 'close_anything', 'StateChange': 'close(drawer_06)'}
9 {'Request': 'close_anything', 'StateChange': 'close(fridge)'}
10 {'Request': 'open_anything', 'StateChange': 'open(drawer_03)'}
11 {'Request': 'open_anything', 'StateChange': 'open(drawer_05)'}
12 {'Request': 'open_anything', 'StateChange': 'open(oven_drawer)'}
13 {'Request': 'open_anything', 'StateChange': 'open(drawer_04)'}
14 {'Request': 'open_anything', 'StateChange': 'open(drawer_01)'}
15 {'Request': 'open_anything',

In [14]:
# Runs a Giskard projection per solution, so it may take a moment
sols = show("taskRequest(Request, StateChange), causes(Motion, StateChange, MotionParam).", limit=3)
# MotionParam = [GoalState, Joint, Handle]

Query: taskRequest(Request, StateChange), causes(Motion, StateChange, MotionParam).
1 {'Request': 'open_fridge', 'StateChange': 'open(fridge)', 'Motion': 'envJointGoal', 'MotionParam': [0.8, 'fridge_door_joint', 'fridge_door_handle']}
2 {'Request': 'open_anything', 'StateChange': 'open(drawer_03)', 'Motion': 'envJointGoal', 'MotionParam': [0.2, 'drawer_03_joint', 'drawer_03_handle']}
3 {'Request': 'open_anything', 'StateChange': 'open(drawer_05)', 'Motion': 'envJointGoal', 'MotionParam': [0.2, 'drawer_05_joint', 'drawer_05_handle']}


In [15]:
sols = show("taskRequest(Request, StateChange), causes(Motion, StateChange, MotionParam), "
            "canPerform(Robot, Motion, MotionParam).", limit=1)

Query: taskRequest(Request, StateChange), causes(Motion, StateChange, MotionParam), canPerform(Robot, Motion, MotionParam).
1 {'Request': 'open_fridge', 'StateChange': 'open(fridge)', 'Motion': 'envJointGoal', 'MotionParam': [0.8, 'fridge_door_joint', 'fridge_door_handle'], 'Robot': 'pr2'}


In [8]:
n_any   = len(list(prolog.query("taskRequest(open_anything, S)")))
n_fridge = len(list(prolog.query("taskRequest(open_fridge, S)")))
print("open_anything solutions:", n_any)
print("open_fridge solutions:  ", n_fridge)

sols = show("taskRequest(open_fridge, StateChange), causes(Motion, StateChange, MotionParam), "
            "canPerform(Robot, Motion, MotionParam).", limit=1)

open_anything solutions: 8
open_fridge solutions:   1
Query: taskRequest(open_fridge, StateChange), causes(Motion, StateChange, MotionParam), canPerform(Robot, Motion, MotionParam).
1 {'StateChange': 'open(fridge)', 'Motion': 'envJointGoal', 'MotionParam': [0.8, 'fridge_door_joint', 'fridge_door_handle'], 'Robot': 'pr2'}


In [16]:
sols = show("taskRequest(open_fridge, StateChange), causes(Motion, StateChange, MotionParam).", limit=1)
goal, joint, handle = sols[0]["MotionParam"]
print("goal joint position:", goal)      # 0.8 (from openState in load_container)
print("joint to move:      ", joint)     # fridge_door_joint
print("handle to grasp:    ", handle)    # fridge_door_handle
# The projected trajectory is published on /trajectory_markers (red line in RViz)

Query: taskRequest(open_fridge, StateChange), causes(Motion, StateChange, MotionParam).
1 {'StateChange': 'open(fridge)', 'Motion': 'envJointGoal', 'MotionParam': [0.8, 'fridge_door_joint', 'fridge_door_handle']}
goal joint position: 0.8
joint to move:       b'fridge_door_joint'
handle to grasp:     b'fridge_door_handle'


In [17]:
# 1) The KB has no friction or force knowledge
print("friction facts:", list(prolog.query("current_predicate(frictionCoefficient/_)")))
print("slip rule:     ", list(prolog.query("current_predicate(slip_edge/_)")))

# 2) Coulomb condition from Lecture 3/4, computed outside the ontology (numbers from the lecture)
def holds(mu, F, m=0.64, a=1.0, g=9.81):
    return 2 * mu * F >= m * (g + a)

for mu, label in [(0.55, "dry"), (0.19, "wet")]:
    print(f"{label}: mu={mu}, 18 N pinch holds in 1 m/s² test lift:", holds(mu, 18))

friction facts: []
slip rule:      []
dry: mu=0.55, 18 N pinch holds in 1 m/s² test lift: True
wet: mu=0.19, 18 N pinch holds in 1 m/s² test lift: False
